# ComplaintRouter — Data Cleaning & Sampling

This notebook prepares CFPB consumer complaint narrative data for ComplaintRouter's product
classification and monetary relief tasks.

**Input:** the archived CFPB bulk file downloaded by `00_download_data.ipynb`
(`data/raw/complaints_2026-08-15_archive.csv.zip`, the last public snapshot that still
contains complaint narratives).

**Output:** `data/processed/complaint_router_modeling_data.csv`, a capped per-product sample of
unique narratives with a cleaned text column, a monetary relief label, and a fixed train/test
split that every modeling notebook shares.

The full file holds millions of complaints, so it is read twice in chunks: the first pass keeps
only small metadata and a hash of each cleaned narrative (enough to deduplicate and sample), and
the second pass retrieves the full narratives for the sampled complaints only.

## 1. Configuration

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 42

raw_path = Path("../data/raw/complaints_2026-08-15_archive.csv.zip")
output_path = Path("../data/processed/complaint_router_modeling_data.csv")

# The proposal keeps complaints filed since August 2023, when the CFPB last revised its
# product and issue categories. Narrative publication ended on August 14, 2026.
START_DATE = "2023-08-01"
END_DATE = "2026-08-15"

CHUNK_SIZE = 250_000

EXPECTED_PRODUCTS = [
    "Checking or savings account",
    "Credit card",
    "Credit reporting or other personal consumer reports",
    "Debt collection",
    "Debt or credit management",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Payday loan, title loan, personal loan, or advance loan",
    "Prepaid card",
    "Student loan",
    "Vehicle loan or lease",
]

print("Raw file:", raw_path, f"({raw_path.stat().st_size / 1e9:,.2f} GB)")

Raw file: ..\data\raw\complaints_2026-08-15_archive.csv.zip (1.42 GB)


## 2. Text Cleaning Function

CFPB narratives are anonymized before publication. Personal details become `XXXX`, dates become
`XX/XX/XXXX` (sometimes with the year kept), and dollar amounts are wrapped in braces such as
`{$200.00}`. Initial TF-IDF analysis showed these placeholders among the highest-weighted terms
for nearly every product, so they are removed before modeling.

Compared with the first version of this notebook, the cleaning now:
- replaces dollar amounts with a single `dollaramount` token instead of breaking them apart
  (the old `\b0{2,}\b` rule turned `{$200.00}` into `{$200. }`);
- removes whole redacted dates rather than leaving `/ /` fragments behind.

The same function is copied into the Streamlit demo so that new complaints are cleaned exactly
the same way at prediction time.

In [3]:
AMOUNT_PATTERN = re.compile(r"\{\$?[\d,]*(?:\.\d*)?\}")
DATE_PATTERN = re.compile(r"\b(?:x{2}|\d{1,2})/(?:x{2}|\d{1,2})/(?:x{2,4}|\d{2,4}|year>?)")
REDACTION_PATTERN = re.compile(r"\bx{2,}\b")
SYMBOL_PATTERN = re.compile(r"[{}<>/\\]")
WHITESPACE_PATTERN = re.compile(r"\s+")


def clean_narrative(text):
    text = str(text).lower()
    text = AMOUNT_PATTERN.sub(" dollaramount ", text)
    text = DATE_PATTERN.sub(" ", text)
    text = REDACTION_PATTERN.sub(" ", text)
    text = SYMBOL_PATTERN.sub(" ", text)
    text = WHITESPACE_PATTERN.sub(" ", text).strip()
    return text


examples = [
    "I am seeking assistance in recovering {$200.00} from XXXX on XX/XX/2025.",
    "fraudulent inquiries XXXX XXXX XXXX, XXXX XXXX XXXX XXX XX/XX/year> and/or 15 USC 1681i",
    "XXXX XXXX",
]

for example in examples:
    print(f"{example!r}\n  -> {clean_narrative(example)!r}\n")

'I am seeking assistance in recovering {$200.00} from XXXX on XX/XX/2025.'
  -> 'i am seeking assistance in recovering dollaramount from on .'

'fraudulent inquiries XXXX XXXX XXXX, XXXX XXXX XXXX XXX XX/XX/year> and/or 15 USC 1681i'
  -> 'fraudulent inquiries , and or 15 usc 1681i'

'XXXX XXXX'
  -> ''



## 3. Pass 1 — Scan the Full Database

Each chunk is filtered to complaints that have a narrative and fall within the date window.
Narratives are cleaned and hashed, and only the hashes and a few metadata columns are kept in
memory. Narratives that are fewer than three words after cleaning are dropped: they carry almost
no information, and very short strings such as `"na"` would be read back as missing values.

In [4]:
pass1_columns = [
    "Date received",
    "Product",
    "Consumer complaint narrative",
    "Complaint ID",
]

chunks = []
narratives_by_year = pd.Series(dtype="int64")
rows_scanned = 0
start = time.perf_counter()

for chunk in pd.read_csv(raw_path, usecols=pass1_columns, dtype=str, chunksize=CHUNK_SIZE):
    rows_scanned += len(chunk)

    chunk = chunk.dropna(subset=["Consumer complaint narrative"])
    narratives_by_year = narratives_by_year.add(
        chunk["Date received"].str[:4].value_counts(), fill_value=0
    )

    chunk = chunk[chunk["Date received"].between(START_DATE, END_DATE)]
    cleaned = chunk["Consumer complaint narrative"].map(clean_narrative)

    chunks.append(pd.DataFrame({
        "Complaint ID": chunk["Complaint ID"].astype("int64").to_numpy(),
        "Date received": chunk["Date received"].to_numpy(),
        "Product": chunk["Product"].to_numpy(),
        "raw_hash": pd.util.hash_pandas_object(
            chunk["Consumer complaint narrative"], index=False
        ).to_numpy(),
        "clean_hash": pd.util.hash_pandas_object(cleaned, index=False).to_numpy(),
        "clean_words": cleaned.str.split().str.len().to_numpy(),
    }))

narratives = pd.concat(chunks, ignore_index=True)
del chunks

print(f"Rows scanned: {rows_scanned:,} in {time.perf_counter() - start:,.0f} seconds")
print(f"Narratives between {START_DATE} and {END_DATE}: {len(narratives):,}")
print()
print("Narratives published by year received (all dates):")
print(narratives_by_year.astype(int).sort_index().to_string())

Rows scanned: 17,119,590 in 424 seconds
Narratives between 2023-08-01 and 2026-08-15: 2,355,520

Narratives published by year received (all dates):
Date received
2015      54723
2016      77766
2017     115311
2018     118393
2019     124843
2020     174297
2021     203570
2022     337273
2023     487410
2024     814384
2025    1222056
2026     107158


In [5]:
print("Date range:", narratives["Date received"].min(), "to", narratives["Date received"].max())

too_short = narratives["clean_words"] < 3
print(f"Narratives under three words after cleaning: {too_short.sum():,}")

narratives = narratives[~too_short].copy()
print(f"Remaining narratives: {len(narratives):,}")

Date range: 2023-08-01 to 2026-07-27
Narratives under three words after cleaning: 1,191
Remaining narratives: 2,354,329


## 4. Validate Product Categories

Only the 11 product categories introduced in the CFPB's 2023 revision are kept. Any complaint
still filed under an older product name is listed by month and removed.

In [6]:
print("Products in the date window:")
print(narratives["Product"].value_counts().to_string())

unexpected = ~narratives["Product"].isin(EXPECTED_PRODUCTS)
print(f"\nComplaints with an unexpected product name: {unexpected.sum():,}")

if unexpected.any():
    print(
        narratives.loc[unexpected]
        .assign(month=narratives.loc[unexpected, "Date received"].str[:7])
        .groupby(["month", "Product"])
        .size()
        .to_string()
    )

narratives = narratives[~unexpected].copy()
print(f"\nNarratives after product validation: {len(narratives):,}")

Products in the date window:
Product
Credit reporting or other personal consumer reports                             1670723
Debt collection                                                                  221706
Checking or savings account                                                      110802
Credit card                                                                      107984
Money transfer, virtual currency, or money service                                87240
Mortgage                                                                          38143
Vehicle loan or lease                                                             27466
Credit reporting, credit repair services, or other personal consumer reports      26732
Student loan                                                                      26671
Payday loan, title loan, personal loan, or advance loan                           18202
Prepaid card                                                                      1


Complaints with an unexpected product name: 29,818
month    Product                                                                     
2023-08  Credit card or prepaid card                                                      2806
         Credit reporting, credit repair services, or other personal consumer reports    26732
         Payday loan, title loan, or personal loan                                         280

Narratives after product validation: 2,324,511


## 5. Remove Duplicate and Ambiguous Narratives

A large share of narratives are copies of the same template letter (for example, credit-repair
dispute letters sent to all three credit bureaus). Duplicates are identified on the **cleaned**
text, which also catches copies that differ only in capitalization or redacted details. Exact
raw-text duplicates are reported for comparison with the team's first run.

A narrative filed under more than one product is ambiguous for product classification, so every
copy of it is removed. For the remaining duplicates the earliest complaint is kept.

In [7]:
print(f"Narratives: {len(narratives):,}")
print(f"Exact duplicate narratives (raw text): {narratives['raw_hash'].duplicated().sum():,}")
print(f"Duplicate narratives after cleaning: {narratives['clean_hash'].duplicated().sum():,}")

print("\nCopies in the 10 largest groups of identical cleaned narratives:")
print(narratives["clean_hash"].value_counts().head(10).tolist())

Narratives: 2,324,511
Exact duplicate narratives (raw text): 1,009,583


Duplicate narratives after cleaning: 1,139,006

Copies in the 10 largest groups of identical cleaned narratives:
[29024, 22409, 18438, 15803, 14380, 12335, 12211, 10144, 10041, 7963]


In [8]:
products_per_narrative = narratives.groupby("clean_hash")["Product"].nunique()
ambiguous_hashes = products_per_narrative[products_per_narrative > 1].index

is_ambiguous = narratives["clean_hash"].isin(ambiguous_hashes)

print(f"Narratives associated with multiple products: {len(ambiguous_hashes):,}")
print(f"Complaints removed as ambiguous: {is_ambiguous.sum():,}")

clean_complaints = (
    narratives[~is_ambiguous]
    .sort_values(["Date received", "Complaint ID"])
    .drop_duplicates(subset=["clean_hash"], keep="first")
    .reset_index(drop=True)
)

print(f"\nFinal unique, unambiguous narratives: {len(clean_complaints):,}")
print("(The team's first run on the CFPB exports: 2,323,440 narratives -> 1,313,972 unique.)")

Narratives associated with multiple products: 1,924
Complaints removed as ambiguous: 282,505



Final unique, unambiguous narratives: 1,183,581
(The team's first run on the CFPB exports: 2,323,440 narratives -> 1,313,972 unique.)


## 6. Product Distribution After Cleaning

This is the real distribution of unique narratives. Credit reporting dominates, which is why the
modeling sample below caps each product.

In [9]:
population_counts = clean_complaints["Product"].value_counts()

population_summary = pd.DataFrame({
    "Unique narratives": population_counts,
    "Share (%)": (population_counts / population_counts.sum() * 100).round(2),
})

population_summary

,Unique narratives,Share (%)
Product,,
Credit reporting or other personal consumer reports,630695,53.29
Debt collection,162266,13.71
Checking or savings account,106278,8.98
Credit card,97804,8.26
"Money transfer, virtual currency, or money service",61378,5.19
Mortgage,38096,3.22
Vehicle loan or lease,27199,2.30
Student loan,26425,2.23
"Payday loan, title loan, personal loan, or advance loan",18068,1.53


## 7. Create Modeling Sample

The sample uses the same per-product caps as the team's first run (162,500 complaints in total).
Capping keeps training manageable and stops credit reporting from dominating, while smaller
products keep enough examples to learn from. Because of this rebalancing, test-set metrics
describe performance on a more balanced mix of products than the CFPB actually receives; this
should be stated in the report.

In [10]:
sample_sizes = {
    "Credit reporting or other personal consumer reports": 35000,
    "Debt collection": 25000,
    "Checking or savings account": 20000,
    "Credit card": 20000,
    "Money transfer, virtual currency, or money service": 15000,
    "Mortgage": 10000,
    "Vehicle loan or lease": 10000,
    "Student loan": 10000,
    "Payday loan, title loan, personal loan, or advance loan": 7500,
    "Prepaid card": 5000,
    "Debt or credit management": 5000
}

In [11]:
sampled_dfs = []

for product, target_size in sample_sizes.items():
    product_data = clean_complaints[
        clean_complaints["Product"] == product
    ]

    # Never request more records than actually exist
    n = min(target_size, len(product_data))

    sampled = product_data.sample(
        n=n,
        random_state=RANDOM_STATE
    )

    sampled_dfs.append(sampled)

sample_index = pd.concat(
    sampled_dfs,
    ignore_index=True
)

# Shuffle the combined sample
sample_index = sample_index.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

sample_comparison = pd.DataFrame({
    "Sample": sample_index["Product"].value_counts(),
    "Sample share (%)": (sample_index["Product"].value_counts(normalize=True) * 100).round(2),
    "Population share (%)": population_summary["Share (%)"],
}).sort_values("Sample", ascending=False)

print("Final modeling sample:", f"{len(sample_index):,}")
sample_comparison

Final modeling sample: 162,500


,Sample,Sample share (%),Population share (%)
Product,,,
Credit reporting or other personal consumer reports,35000,21.54,53.29
Debt collection,25000,15.38,13.71
Checking or savings account,20000,12.31,8.98
Credit card,20000,12.31,8.26
"Money transfer, virtual currency, or money service",15000,9.23,5.19
Mortgage,10000,6.15,3.22
Vehicle loan or lease,10000,6.15,2.30
Student loan,10000,6.15,2.23
"Payday loan, title loan, personal loan, or advance loan",7500,4.62,1.53


## 8. Pass 2 — Retrieve Narratives for the Sample

The file is read a second time, keeping the full set of analysis columns for the sampled
complaints only.

In [12]:
columns_to_keep = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Company",
    "Company response to consumer",
    "Timely response?",
    "Complaint ID"
]

sampled_ids = sample_index["Complaint ID"].to_numpy()
details = []
start = time.perf_counter()

for chunk in pd.read_csv(raw_path, usecols=columns_to_keep, dtype=str, chunksize=CHUNK_SIZE):
    chunk = chunk[chunk["Complaint ID"].astype("int64").isin(sampled_ids)]
    details.append(chunk)

details = pd.concat(details, ignore_index=True)
details["Complaint ID"] = details["Complaint ID"].astype("int64")

modeling_data = (
    sample_index[["Complaint ID", "clean_hash"]]
    .merge(details, on="Complaint ID", how="left", validate="one_to_one")
)

print(f"Retrieved {len(details):,} complaints in {time.perf_counter() - start:,.0f} seconds")
print("Missing narratives:", modeling_data["Consumer complaint narrative"].isna().sum())

Retrieved 162,500 complaints in 120 seconds
Missing narratives: 0


In [13]:
modeling_data["clean_narrative"] = (
    modeling_data["Consumer complaint narrative"]
    .map(clean_narrative)
)

# The cleaned text must hash to the same value seen in pass 1
rehashed = pd.util.hash_pandas_object(modeling_data["clean_narrative"], index=False).to_numpy()
print("Cleaned text matches pass 1:", bool((rehashed == modeling_data["clean_hash"].to_numpy()).all()))

modeling_data = modeling_data.drop(columns="clean_hash")
print(modeling_data.shape)

Cleaned text matches pass 1: True
(162500, 11)


## 9. Monetary Relief Label

The proposal's secondary task predicts whether a complaint closes with **monetary relief**.
Complaints that were still in progress, or that the company did not answer on time, have no
final outcome yet, so their label is left missing and they are excluded from the relief task.

In [14]:
pending_responses = ["In progress", "Untimely response"]
response = modeling_data["Company response to consumer"]

modeling_data["monetary_relief"] = np.where(
    response.isna() | response.isin(pending_responses),
    np.nan,
    (response == "Closed with monetary relief").astype(float),
)

print(
    modeling_data.groupby("Company response to consumer", dropna=False)["monetary_relief"]
    .agg(["size", "mean"])
    .rename(columns={"size": "Complaints", "mean": "monetary_relief"})
)

labeled = modeling_data["monetary_relief"].notna()
print(f"\nComplaints with a final outcome: {labeled.sum():,}")
print(f"Monetary relief rate: {modeling_data.loc[labeled, 'monetary_relief'].mean() * 100:.2f}%")

                                 Complaints  monetary_relief
Company response to consumer                                
Closed with explanation              125907              0.0
Closed with monetary relief            9206              1.0
Closed with non-monetary relief       25480              0.0
In progress                              27              NaN
Untimely response                      1880              NaN

Complaints with a final outcome: 160,593
Monetary relief rate: 5.73%


## 10. Train/Test Split

A single 80/20 split, stratified by product, is stored in a `split` column so that every
modeling notebook trains and evaluates on exactly the same complaints. Hyperparameters are tuned
with cross-validation inside the training set; the test set is only used for final evaluation.

In [15]:
train_index, test_index = train_test_split(
    modeling_data.index,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=modeling_data["Product"]
)

modeling_data["split"] = "train"
modeling_data.loc[test_index, "split"] = "test"

print(modeling_data["split"].value_counts())
print()
print(pd.crosstab(modeling_data["Product"], modeling_data["split"]))

split
train    130000
test      32500
Name: count, dtype: int64



split                                               test  train
Product                                                        
Checking or savings account                         4000  16000
Credit card                                         4000  16000
Credit reporting or other personal consumer rep...  7000  28000
Debt collection                                     5000  20000
Debt or credit management                           1000   4000
Money transfer, virtual currency, or money service  3000  12000
Mortgage                                            2000   8000
Payday loan, title loan, personal loan, or adva...  1500   6000
Prepaid card                                        1000   4000
Student loan                                        2000   8000
Vehicle loan or lease                               2000   8000


## 11. Near-Duplicate Check

Exact duplicates were removed, but template letters with small edits (a different account name
at the end, for example) can still appear in both the training and test sets, which would make
test scores look better than they really are. As a quick estimate, this counts test narratives
whose first 200 cleaned characters also start a training narrative. The explainability notebook
measures whether these matches change model accuracy.

In [16]:
prefix = modeling_data["clean_narrative"].str[:200]
train_prefixes = set(prefix[modeling_data["split"] == "train"])

test_prefix_match = prefix[modeling_data["split"] == "test"].isin(train_prefixes)

print(f"Test narratives sharing a 200-character opening with a training narrative: "
      f"{test_prefix_match.sum():,} ({test_prefix_match.mean() * 100:.2f}%)")

Test narratives sharing a 200-character opening with a training narrative: 1,818 (5.59%)


## 12. Cleaning Examples

In [17]:
for i in range(3):
    print("Original:")
    print(modeling_data["Consumer complaint narrative"].iloc[i][:400])
    print("\nCleaned:")
    print(modeling_data["clean_narrative"].iloc[i][:400])
    print("\n" + "-" * 80 + "\n")

Original:
ssue : Failure to provide proper debt validation / inaccurate reporting under the Fair Debt Collection Practices Act ( FDCPA ) and Fair Credit Reporting Act ( FCRA ). 

Details : On XX/XX/2025, I submitted a complaint regarding NCB Management Services reporting an alleged {$4200.00} debt without providing proper validation. In their response to the CFPB, NCB stated they would provide the following

Cleaned:
ssue : failure to provide proper debt validation inaccurate reporting under the fair debt collection practices act ( fdcpa ) and fair credit reporting act ( fcra ). details : on , i submitted a complaint regarding ncb management services reporting an alleged dollaramount debt without providing proper validation. in their response to the cfpb, ncb stated they would provide the following documents :

--------------------------------------------------------------------------------

Original:
Delinquent, derogatory, and collection accounts that should be removed. 
XXXX XXXX ac

## 13. Save Processed Dataset

In [18]:
output_columns = [
    "Complaint ID",
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company",
    "Company response to consumer",
    "Timely response?",
    "Consumer complaint narrative",
    "clean_narrative",
    "monetary_relief",
    "split",
]

output_path.parent.mkdir(parents=True, exist_ok=True)
modeling_data[output_columns].to_csv(output_path, index=False)

print(f"Saved {len(modeling_data):,} rows to:")
print(output_path, f"({output_path.stat().st_size / 1e6:,.0f} MB)")

Saved 162,500 rows to:
..\data\processed\complaint_router_modeling_data.csv (434 MB)


In [19]:
test_load = pd.read_csv(output_path)

print("Saved shape:", test_load.shape)
print("Missing narratives:", test_load["Consumer complaint narrative"].isna().sum())
print("Missing clean narratives:", test_load["clean_narrative"].isna().sum())
print("Number of classes:", test_load["Product"].nunique())
print("Split counts:", test_load["split"].value_counts().to_dict())

Saved shape: (162500, 13)
Missing narratives: 0
Missing clean narratives: 0
Number of classes: 11
Split counts: {'train': 130000, 'test': 32500}


## Summary

| Step | Complaints |
|---|---|
| Rows in the archived CFPB file (all years) | 17,119,590 |
| Narratives received 2023-08-01 to 2026-07-27 | 2,355,520 |
| Under three words after cleaning (dropped) | 1,191 |
| Filed under pre-2023 product names, all in August 2023 (dropped) | 29,818 |
| Duplicate narratives after cleaning (1,009,583 exact raw duplicates) | 1,139,006 |
| Narratives tied to more than one product (1,924 texts, all copies dropped) | 282,505 |
| **Unique, unambiguous narratives** | **1,183,581** |
| **Modeling sample (130,000 train / 32,500 test)** | **162,500** |

- The narrative count closely matches the team's original CFPB exports (2,323,440 narratives), so the
  archived snapshot reproduces the same source data. Deduplicating on cleaned text removes about
  130,000 more template letters than exact matching did, which is why fewer unique narratives remain
  (1.18M vs 1.31M).
- Template letters are extreme: the largest group of identical cleaned narratives has 29,024 copies.
- The real population is 53% credit reporting; the capped sample brings it to 21.5% so smaller
  products are well represented. Test metrics therefore describe a more balanced product mix than
  the CFPB receives.
- **Monetary relief** occurs in 5.73% of the 160,593 complaints with a final outcome; 1,907 untimely
  or in-progress complaints have no label.
- 5.59% of test narratives share their first 200 cleaned characters with a training narrative. These
  near-duplicates could inflate test scores slightly; `04_explainability_error_analysis.ipynb`
  checks whether they do.
- Narratives for complaints received after late July 2026 were never published before the CFPB
  ended narrative publication on August 14, 2026, and 2026 has far fewer narratives overall
  (107,158 vs 1,222,056 in 2025).